# L28 Lab: FMCW Range, notebook

This notebook runs the course's radar chain, `radar_lab.py`, on saved captures. You do not write the chain; you run it, change what each task names, and record what you see in the L28 Lab packet. A readout cell gives the peak frequencies for Tables A to C, and three tasks, (a) to (c), fill Table E, about 30 to 45 minutes in all. Each task changes one line.

**To run it:** unzip the lab's bundle, `L28_lab.zip` from the L28 page, into one folder: it holds this notebook, `radar_lab.py`, and every recorded capture. Save your own captures into the same folder. It needs Python 3 with numpy, scipy, matplotlib, and Jupyter: `python -m pip install numpy scipy matplotlib jupyterlab` installs all four. Run the cells in order from the top.

If your own captures failed, the course's recorded data (`L28_*.npz`) works in every cell.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import radar_lab as rl

# Where the captures are: this folder (the unzipped bundle), or the course's
# radar-data/ folder if this notebook sits beside it.
DATA = Path(".") if Path("L28_corner_2m.npz").exists() or not Path("../radar-data").exists() else Path("../radar-data")

# Your 2 m capture (Procedure B). Change the name to your own file to use your data.
CAPTURE = DATA / "L28_corner_2m.npz"
cap = rl.load_capture(CAPTURE)
print(f"{CAPTURE.name}: {cap['iq'].shape[0]} chirps of {cap['iq'].shape[1]} samples, "
      f"B = {cap['B']/1e6:.0f} MHz, T_ramp = {cap['T_ramp']*1e3:.2f} ms, "
      f"fs = {cap['fs']/1e6:.1f} MSPS, f0 = {cap['f0']/1e3:.1f} kHz")

In [ ]:
# Readouts used by the cells below. Run this cell; you do not need to change it.
def peak(r, p_db, lo=0.5, hi=8.0):
    """Index, range, and level of the strongest point between lo and hi meters."""
    m = (r >= lo) & (r <= hi)
    i = np.flatnonzero(m)[np.argmax(p_db[m])]
    return i, r[i], p_db[i]

def width_3db(x, p_db, i):
    """Width of the peak at index i, 3 dB down, in the units of x."""
    lo = hi = i
    while lo > 0 and p_db[lo - 1] > p_db[i] - 3: lo -= 1
    while hi < len(p_db) - 1 and p_db[hi + 1] > p_db[i] - 3: hi += 1
    return x[hi] - x[lo] + (x[1] - x[0])

def sidelobe(r, p_db, i, span=1.2):
    """Strongest sidelobe within span m of the peak at index i, in dB below it."""
    a = b = i
    while a + 1 < len(p_db) and p_db[a + 1] < p_db[a]: a += 1     # walk down to each first null
    while b > 0 and p_db[b - 1] < p_db[b]: b -= 1
    near = (np.abs(r - r[i]) < span)
    out = near.copy(); out[b:a + 1] = False
    return p_db[out].max() - p_db[i]

def peak_table(c, n_fft=16384, top=4):
    """The strongest peaks of a capture, every chirp's power averaged, Hann window.

    Returns one row per peak, lowest frequency first: frequency in kHz, level in
    dBFS, height above the median floor 8 to 12 m out in dB, and width 3 dB down
    in m. Ranges use the capture's own f0, as the app does.
    """
    X = rl.range_fft(c, "hann", n_fft)
    p = 10 * np.log10(np.mean(np.abs(X) ** 2, axis=0) + 1e-30)
    f = np.fft.fftfreq(n_fft, 1 / c["fs"])
    r = rl.range_axis(c, n_fft)
    cell = rl.C / (2 * c["B"])                                    # one resolution cell, m
    floor = np.median(p[(r > 8) & (r < 12)])
    bin_m = (r[1] - r[0]) * n_fft / c["iq"].shape[1]                # one FFT bin, m
    cand = np.flatnonzero((f[1:-1] >= 0) & (p[1:-1] > p[:-2]) & (p[1:-1] >= p[2:])
                          & (p[1:-1] > floor + 15)) + 1
    kept = []
    for k in cand[np.argsort(p[cand])[::-1]]:                     # strongest first; skip a
        if (all(abs(r[k] - r[j]) > 3 * cell for j in kept)         # stronger peak's sidelobes,
                and bin_m < width_3db(r, p, k) < 4 * bin_m):       # under a bin wide, and the
            kept.append(k)                                         # shoulders between peaks
        if len(kept) == top: break
    return [(f[k] / 1e3, p[k], p[k] - floor, width_3db(r, p, k)) for k in sorted(kept)]

## Peak readout (Tables A to C)

The app shows range; the packet's Tables A to C also want each peak's **frequency**. This cell lists the strongest peaks of any capture: frequency in kHz, level, height above the noise floor, and width 3 dB down. It searches every frequency, the 0 m end included, so it reads the leakage and a plate against the array as well as a target in the room.

Set `NAME` to a capture and run the cell, once for each capture a table asks about: your own files, or the recorded ones.

**Check** on the recorded data: `L28_clear.npz` shows the leakage at 101.40 kHz and the wall near 119.8 kHz. `L28_cal_0m.npz` shows the plate at 101.48 kHz, 10 dB above the leakage it covers. `L28_corner_2m.npz` shows the leakage, the corner at 108.11 kHz, and the wall. A peak that does not move from capture to capture is the leakage or the wall.

A capture the app saves holds the receiver's raw counts, not the recorded data's full-scale units, so its levels print tens of dB higher. Compare the differences between levels, which the heights above the floor are, not the levels themselves.

In [ ]:
NAME = "L28_cal_0m.npz"        # the capture to read: change it for each table row

rows = peak_table(rl.load_capture(DATA / NAME))
print(f"{NAME}:")
for f_khz, level, height, width in rows:
    print(f"  {f_khz:8.2f} kHz  {level:6.1f} dBFS  {height:5.1f} dB above the floor  "
          f"{width:.2f} m wide 3 dB down")

## Task (a): Hann against rectangular

The cell below draws one chirp's range profile and reads three numbers: the plate's peak, its strongest sidelobe, and the floor near 10 m.

1. Run it as it stands (`WINDOW = "hann"`) and record the three numbers in the Hann column of Table E.
2. **Change one line:** set `WINDOW = None`, the rectangular window, and run it again. Record the rectangular column.

**Check**, on `L28_corner_2m.npz`: with Hann the peak reads 2.01 m at about $-19.6$ dBFS, the strongest sidelobe is about 31 dB below it, and the floor near 10 m is about $-86$ dBFS. With the rectangular window the peak level hardly changes, because `radar_lab` divides every FFT by the sum of its window, the sidelobe rises to 12 to 13 dB below the peak, and the floor near 10 m rises by about 30 dB, to about $-56$ dBFS. Look at the plot and decide whether that rise is noise or something else: Part 3 of the packet asks you.

In [ ]:
# Task (a)
WINDOW = "hann"          # TASK (a): change to None for the rectangular window

r, p_db = rl.range_profile(cap, chirp=0, window=WINDOW, n_fft=4096)   # zero-padded, so the peak is easy to read
i, r_pk, peak_db = peak(r, p_db)
sidelobe_db = sidelobe(r, p_db, i)
floor_db = np.median(p_db[(r > 8) & (r < 12)])
print(f"window {WINDOW}: peak {r_pk:.2f} m at {peak_db:.1f} dBFS, "
      f"strongest sidelobe {sidelobe_db:.1f} dBc, floor near 10 m {floor_db:.1f} dBFS")

plt.figure(figsize=(8, 3))
plt.plot(r, p_db)
plt.xlim(0, 12); plt.ylim(-100, 0)
plt.xlabel("range (m)"); plt.ylabel("power (dBFS)"); plt.title(f"one chirp, window {WINDOW}")
plt.grid(alpha=0.3); plt.show()

## Task (b): does zero-padding resolve two targets?

`L28_two_corners.npz` holds two corners 0.20 m apart, at 2.0 and 2.2 m: closer than one range cell. The cell draws them first with no padding, one point per FFT bin: 540 points, 0.33 m apart.

1. Run it as it stands (`N_FFT = None`) and count the peaks between 1.5 and 2.7 m.
2. **Change one line**, the `N_FFT` line in this cell: set `N_FFT = 4096` and run it again. Count the peaks and read the width 3 dB down.

**Check:** one peak both times, between the two corners: at 2.25 m on the coarse grid, and at 2.10 m at 4096 points. Its width 3 dB down prints as 0.67 m with no padding, where the grid is two bins across it, and 0.53 m at 4096 points; one corner alone is 0.48 m wide. Record both widths in Table E. Zero-padding draws the same spectrum on a grid 7.6 times finer; it cannot split two beats that differ by less than one cycle a chirp.

In [ ]:
# Task (b)
N_FFT = None             # TASK (b): change to 4096

two = rl.load_capture(DATA / "L28_two_corners.npz")
r2, p2 = rl.range_profile(two, window="hann", n_fft=N_FFT)
m = (r2 > 1.5) & (r2 < 2.7)
peaks = np.flatnonzero(m[1:-1] & (p2[1:-1] > p2[:-2]) & (p2[1:-1] > p2[2:]) & (p2[1:-1] > p2[m].max() - 6)) + 1
n_peaks = len(peaks)
i2, r_pk2, _ = peak(r2, p2, 1.5, 2.7)
width_m = width_3db(r2, p2, i2)
print(f"n_fft {N_FFT}: {n_peaks} peak(s) between 1.5 and 2.7 m, at {np.round(r2[peaks], 2)} m; "
      f"width 3 dB down {width_m:.2f} m")

plt.figure(figsize=(8, 3))
plt.plot(r2, p2, ".-" if N_FFT is None else "-")
plt.axvline(2.0, color="gray", ls=":"); plt.axvline(2.2, color="gray", ls=":")
plt.xlim(0.5, 4); plt.ylim(-80, 0)
plt.xlabel("range (m)"); plt.ylabel("power (dBFS)"); plt.title(f"two corners 0.20 m apart, n_fft {N_FFT}")
plt.grid(alpha=0.3); plt.show()

## Task (c): range from your own 0 m frequency

`range_profile` converts frequency to range for you, using the capture's `f0`. This task does the conversion by hand, from the frequency of the plate's peak, with **your** 0 m frequency from Table A.

1. Set `F0` to your Table A value, in hertz.
2. **Change one line:** replace `None` in `R = None` with the range of the peak, from `f_peak`, `F0`, and the capture's `B` and `T_ramp` (the range equation of the lesson's range FFT frame). Use `rl.C` for the speed of light.

**Check**, on `L28_corner_2m.npz` with `F0 = 101.48e3`, the recorded plate's 0 m frequency: the peak is at 108.11 kHz, 6.63 kHz above `F0`, and `R` is 1.99 m. `R` agrees with the tape-measure range to within about 0.1 m, and with the peak the profile in task (a) found. Record `R` in Table E. If it is off by a fixed amount at every range, look at `F0` first: 1 kHz of error in $f_0$ moves every range by 0.30 m.

In [ ]:
# Task (c)
F0 = cap["f0"]           # TASK (c), step 1: your Table A 0 m frequency, in Hz

X = rl.range_fft(cap, window="hann", n_fft=4096)[0]
f = np.fft.fftfreq(4096, 1 / cap["fs"])
k = 2 * cap["B"] / (rl.C * cap["T_ramp"])           # Hz of beat per meter
search = (f > F0 + 0.5 * k) & (f < cap["fs"] / 2)  # beats from 0.5 m out
f_peak = f[search][np.argmax(np.abs(X[search]))]

R = None                 # TASK (c), step 2: the range of the peak, in m

print(f"peak at {f_peak/1e3:.2f} kHz, {(f_peak - F0)/1e3:.2f} kHz above F0; R = {R}")

## The waterfall

No task here: run it for the waterfall plot the packet asks for. `L28_walker.npz` is the course's 10 s record of a person walking from 1 m out to 4 m and back, twice; the app's waterfall of your own walk is the screenshot Procedure D takes.

**Check:** a vertical line at 0 m, the leakage, one at 5.5 m, the wall, and a zigzag between 1 and 4 m whose legs each take 3 s.

In [ ]:
WALK = DATA / "L28_walker.npz"
walk = rl.load_capture(WALK)
rw, tw, w_db = rl.waterfall(walk)
keep = rw <= 6
plt.figure(figsize=(8, 4))
plt.pcolormesh(rw[keep], tw, w_db[:, keep], vmin=-65, vmax=-25, cmap="Blues", shading="auto")
plt.xlabel("range (m)"); plt.ylabel("time (s)"); plt.title(WALK.name)
plt.colorbar(label="dBFS"); plt.show()

## Before you close

Save the notebook with your changes and your plots, and upload the `.ipynb` with the packet. Leave the task cells' names and variables as they are. Keep it, `radar_lab.py`, and your captures together in your Module 4 folder. Anything you do beyond these three tasks, such as your own plot, another window, or another capture, belongs in your report's Extra Effort.